In [1]:
import pandas as pd

In [3]:
users = pd.read_csv("../dataset/users.csv")

products = pd.read_csv("../dataset/products.csv")

interactions = pd.read_csv("../dataset/interactions.csv")

In [4]:
print(users)
print(products)
print(interactions)

   user_id education             interests
0        1       BCA     technology coding
1        2  Commerce      finance business
2        3      Arts        design fashion
3        4     BTech        programming AI
4        5       MBA  management marketing
   product_id           product_name     category  \
0         101          Gaming Laptop  electronics   
1         102           Finance Book        books   
2         103         Graphic Tablet  electronics   
3         104     Programming Course    education   
4         105  Marketing Masterclass    education   
5         106             AI Toolkit  electronics   

                                         tags  
0       technology coding programming student  
1        finance business commerce investment  
2               design art creativity drawing  
3      coding technology software development  
4        marketing business management growth  
5  AI machine-learning programming technology  
   user_id  product_id    action
0

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
users["profile"] = (
    users["education"] + " " +
    users["interests"]
)

print(users[["user_id", "profile"]])

   user_id                    profile
0        1      BCA technology coding
1        2  Commerce finance business
2        3        Arts design fashion
3        4       BTech programming AI
4        5   MBA management marketing


In [7]:
vectorizer = TfidfVectorizer()

product_vectors = vectorizer.fit_transform(
    products["tags"]
)

print(product_vectors.shape)

(6, 20)


In [8]:
user_vectors = vectorizer.transform(
    users["profile"]
)

print(user_vectors.shape)


(5, 20)


In [9]:
similarity = cosine_similarity(
    user_vectors,
    product_vectors
)

print(similarity)

[[0.6386017  0.         0.         0.60450463 0.         0.219188  ]
 [0.         0.85305348 0.         0.         0.21464157 0.        ]
 [0.         0.         0.5        0.         0.         0.        ]
 [0.30940431 0.         0.         0.         0.         0.63468627]
 [0.         0.         0.         0.         0.73796985 0.        ]]


In [10]:
for i, user in users.iterrows():

    scores = similarity[i]

    sorted_indices = scores.argsort()[::-1]

    top_products = sorted_indices[:3]

    print(f"\nRecommendations for User {user['user_id']}:")

    for index in top_products:

        product_name = products.iloc[index]["product_name"]

        score = scores[index]

        print(f"{product_name} | Score: {score:.2f}")


Recommendations for User 1:
Gaming Laptop | Score: 0.64
Programming Course | Score: 0.60
AI Toolkit | Score: 0.22

Recommendations for User 2:
Finance Book | Score: 0.85
Marketing Masterclass | Score: 0.21
AI Toolkit | Score: 0.00

Recommendations for User 3:
Graphic Tablet | Score: 0.50
AI Toolkit | Score: 0.00
Marketing Masterclass | Score: 0.00

Recommendations for User 4:
AI Toolkit | Score: 0.63
Gaming Laptop | Score: 0.31
Marketing Masterclass | Score: 0.00

Recommendations for User 5:
Marketing Masterclass | Score: 0.74
AI Toolkit | Score: 0.00
Programming Course | Score: 0.00


In [11]:
interaction_data = interactions.merge(
    products,
    on="product_id"
)

print(interaction_data.head())

   user_id  product_id    action        product_name     category  \
0        1         101     click       Gaming Laptop  electronics   
1        1         104  purchase  Programming Course    education   
2        2         102     click        Finance Book        books   
3        3         103  purchase      Graphic Tablet  electronics   
4        4         106     click          AI Toolkit  electronics   

                                         tags  
0       technology coding programming student  
1      coding technology software development  
2        finance business commerce investment  
3               design art creativity drawing  
4  AI machine-learning programming technology  


In [12]:
interaction_data = interactions.merge(
    products,
    on="product_id"
)

print(interaction_data.head())

   user_id  product_id    action        product_name     category  \
0        1         101     click       Gaming Laptop  electronics   
1        1         104  purchase  Programming Course    education   
2        2         102     click        Finance Book        books   
3        3         103  purchase      Graphic Tablet  electronics   
4        4         106     click          AI Toolkit  electronics   

                                         tags  
0       technology coding programming student  
1      coding technology software development  
2        finance business commerce investment  
3               design art creativity drawing  
4  AI machine-learning programming technology  


In [13]:
user_behavior = interaction_data.groupby(
    "user_id"
)["tags"].apply(
    lambda x: " ".join(x)
)

print(user_behavior)

user_id
1    technology coding programming student coding t...
2                 finance business commerce investment
3                        design art creativity drawing
4           AI machine-learning programming technology
5                 marketing business management growth
Name: tags, dtype: object


In [14]:
users["behavior"] = users["user_id"].map(
    user_behavior
)

users["behavior"] = users["behavior"].fillna("")

print(users[[
    "user_id",
    "behavior"
]])


   user_id                                           behavior
0        1  technology coding programming student coding t...
1        2               finance business commerce investment
2        3                      design art creativity drawing
3        4         AI machine-learning programming technology
4        5               marketing business management growth


In [15]:
users["final_profile"] = (
    users["education"] + " " +
    users["interests"] + " " +
    users["behavior"]
)

print(users[[
    "user_id",
    "final_profile"
]])

   user_id                                      final_profile
0        1  BCA technology coding technology coding progra...
1        2  Commerce finance business finance business com...
2        3  Arts design fashion design art creativity drawing
3        4  BTech programming AI AI machine-learning progr...
4        5  MBA management marketing marketing business ma...


In [16]:
new_user_vectors = vectorizer.transform(
    users["final_profile"]
)

In [17]:
new_similarity = cosine_similarity(
    new_user_vectors,
    product_vectors
)

print(new_similarity)


[[0.81436337 0.         0.         0.82012972 0.         0.2764278 ]
 [0.         0.96837419 0.         0.         0.20525555 0.        ]
 [0.         0.         0.94491118 0.         0.         0.        ]
 [0.35847168 0.         0.         0.08915992 0.         0.94396801]
 [0.         0.11282331 0.         0.         0.95175379 0.        ]]


In [18]:
for i, user in users.iterrows():

    scores = new_similarity[i]

    sorted_indices = scores.argsort()[::-1]

    top_products = sorted_indices[:3]

    print(f"\nUpdated Recommendations for User {user['user_id']}:")

    for index in top_products:

        product_name = products.iloc[index]["product_name"]

        score = scores[index]

        print(f"{product_name} | Score: {score:.2f}")


Updated Recommendations for User 1:
Programming Course | Score: 0.82
Gaming Laptop | Score: 0.81
AI Toolkit | Score: 0.28

Updated Recommendations for User 2:
Finance Book | Score: 0.97
Marketing Masterclass | Score: 0.21
AI Toolkit | Score: 0.00

Updated Recommendations for User 3:
Graphic Tablet | Score: 0.94
AI Toolkit | Score: 0.00
Marketing Masterclass | Score: 0.00

Updated Recommendations for User 4:
AI Toolkit | Score: 0.94
Gaming Laptop | Score: 0.36
Programming Course | Score: 0.09

Updated Recommendations for User 5:
Marketing Masterclass | Score: 0.95
Finance Book | Score: 0.11
AI Toolkit | Score: 0.00
